In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

# 从项目根目录或 examples 目录启动均可；统一以项目根目录运行。
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "mtp_initializer").is_dir() and (p / "PROJECT.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("请从 scipykit 项目根目录或 examples 目录启动 notebook")
os.chdir(PROJECT_ROOT)
# 根目录用于本地脚本；父目录用于 import scipykit。同步对子进程生效。
python_paths = [str(PROJECT_ROOT), str(PROJECT_ROOT.parent)]
for path in reversed(python_paths):
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = os.pathsep.join(
    dict.fromkeys(python_paths + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p])
)
os.environ["NOTEBOOK_NAME"] = "03_日志快速上手"
print("项目路径:", PROJECT_ROOT)
print("当前解释器:", sys.executable)


# 把日志当作带级别的 print
默认只输出终端。多个位置参数按空格拼接，字符串花括号保持原样，推荐直接用 f-string。默认显示调用文件/模块、函数、行号；notebook 中显示 IPython 单元格来源。

In [ ]:
from scipykit.log import *
configure_log(preset="source", level="DEBUG", colors=True)
log("开始实验", {"batch_size": 16, "learning_rate": 0.001})
log.debug("调试信息", [1, 2, 3])
log.success("初始化完成")
log.warning("样本数量偏少", count=12)
log.error("这里只演示错误级别，不抛异常")
log("字面量 {value} 不会被隐式替换")

## 从临时打印切换到落盘
`file=路径` 只保存这一句；`record_to` 保存一个代码段；`log_to` 开启持续记录，直到 `stop_log`。已开启文件记录时，`file=False` 可让某一句只出现在终端。每条记录仍受对应终端/文件的级别过滤。

In [ ]:
logs = Path("assets/03_日志快速上手/logs")
log("临时看看")
log("只把本句也写入文件", file=logs / "one.log")
with record_to(logs / "stage.log", mode="w"):
    log("进入阶段")
    log("这句临时输出，不落盘", file=False)
    log.success("阶段完成")
log_to(logs / "run.log", rotation="10 MB", retention="7 days", mode="w")
log("开始持续保存")
log.warning("持续记录中的警告")
stop_log()
log("已恢复只输出终端")
print((logs / "stage.log").read_text())

## 用少量业务信息标记职责
`get_logger("组件", run="实验名")` 绑定职责；`log_context` 为当前线程/异步上下文补充信息。手动标记不会替换真实调用文件和函数。多层封装可通过 `depth=1` 再向外追溯一层。

In [ ]:
loader_log = get_logger("dataset", run="exp01")
def load_batch(index):
    loader_log.info("读取批次", batch=index)
    return [index] * 3

with log_context(stage="validation", owner="vision"):
    batch = load_batch(2)
    log("批次内容", batch)
log("上下文已恢复")

## 异常与结构化文件
`log.exception` 在 except 中记录 traceback，不吞掉异常，也不自动重新抛出。JSONL 每行一个对象，可按 `file/function/line` 和 `extra` 检索。文件始终使用 UTF-8，终端颜色与控制码会清理。

In [ ]:
import json
with record_to(logs / "records.jsonl", serialize=True, mode="w"):
    with log_context(run="exp01"):
        log.success("评估结果", accuracy=0.93)
    try:
        raise ValueError("演示异常")
    except ValueError:
        log.exception("本次输入无效")
records = [json.loads(line) for line in (logs / "records.jsonl").read_text().splitlines()]
print("第一条记录:", records[0])
assert records[0]["extra"]["accuracy"] == 0.93